## 이진분류(이직여부예측) 

In [131]:
#기본
import numpy as np
import pandas as pd

In [132]:
#스케일링
from sklearn.preprocessing import MinMaxScaler
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import MaxAbsScaler
from sklearn.preprocessing import RobustScaler

#데이터나누기
from sklearn.model_selection import train_test_split

In [133]:
#모델
from sklearn.ensemble import RandomForestClassifier

In [134]:
#인코딩
from sklearn.preprocessing import LabelEncoder

In [135]:
#평가
from sklearn.metrics import roc_auc_score

In [33]:
train = pd.read_csv('hr_train.csv')
test = pd.read_csv('hr_test.csv')

### 탐색적 데이터 분석

In [34]:
print("train :", train.shape)
print("test :", test.shape)

train : (15326, 14)
test : (3832, 13)


In [35]:
train.head()

,enrollee_id,city,city_development_index,gender,relevent_experience,enrolled_university,education_level,major_discipline,experience,company_size,company_type,last_new_job,training_hours,target
0,30266,city_84,0.698,Male,No relevent experience,Full time course,High School,NaN,1,100-500,Pvt Ltd,never,15,0.0
1,13254,city_16,0.910,Male,Has relevent experience,no_enrollment,Graduate,STEM,15,50-99,Pvt Ltd,1,4,0.0
2,31675,city_21,0.624,Male,No relevent experience,Full time course,High School,NaN,3,NaN,NaN,never,12,0.0
3,30804,city_104,0.924,Male,No relevent experience,no_enrollment,Graduate,Arts,>20,NaN,NaN,2,258,1.0
4,18269,city_143,0.740,Male,Has relevent experience,no_enrollment,Graduate,STEM,12,50-99,Pvt Ltd,1,3,0.0


In [36]:
test.head()

,enrollee_id,city,city_development_index,gender,relevent_experience,enrolled_university,education_level,major_discipline,experience,company_size,company_type,last_new_job,training_hours
0,23044,city_136,0.897,Female,No relevent experience,Full time course,Masters,STEM,8,100-500,Public Sector,1,35
1,29611,city_21,0.624,NaN,Has relevent experience,no_enrollment,Graduate,STEM,16,<10,Pvt Ltd,>4,134
2,22483,city_21,0.624,NaN,Has relevent experience,no_enrollment,Graduate,STEM,5,10000+,Pvt Ltd,1,92
3,2660,city_103,0.920,Male,Has relevent experience,no_enrollment,Masters,STEM,>20,100-500,Funded Startup,>4,6
4,29216,city_65,0.802,Male,Has relevent experience,no_enrollment,Graduate,STEM,15,<10,Pvt Ltd,>4,61


In [37]:
train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 15326 entries, 0 to 15325
Data columns (total 14 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   enrollee_id             15326 non-null  int64  
 1   city                    15326 non-null  object 
 2   city_development_index  15326 non-null  float64
 3   gender                  11750 non-null  object 
 4   relevent_experience     15326 non-null  object 
 5   enrolled_university     15012 non-null  object 
 6   education_level         14961 non-null  object 
 7   major_discipline        13045 non-null  object 
 8   experience              15272 non-null  object 
 9   company_size            10539 non-null  object 
 10  company_type            10383 non-null  object 
 11  last_new_job            14984 non-null  object 
 12  training_hours          15326 non-null  int64  
 13  target                  15326 non-null  float64
dtypes: float64(2), int64(2), object(10)
me

In [38]:
train.describe()

,enrollee_id,city_development_index,training_hours,target
count,15326.000000,15326.000000,15326.000000,15326.000000
mean,16890.840663,0.828909,65.110596,0.248532
std,9615.509411,0.123372,59.610006,0.432176
min,2.000000,0.448000,1.000000,0.000000
25%,8582.500000,0.740000,23.000000,0.000000
50%,16982.000000,0.903000,47.000000,0.000000
75%,25186.250000,0.920000,88.000000,0.000000
max,33380.000000,0.949000,336.000000,1.000000


In [39]:
train.describe(include='O')

,city,gender,relevent_experience,enrolled_university,education_level,major_discipline,experience,company_size,company_type,last_new_job
count,15326,11750,15326,15012,14961,13045,15272,10539,10383,14984
unique,123,3,2,3,5,6,22,8,6,6
top,city_103,Male,Has relevent experience,no_enrollment,Graduate,STEM,>20,50-99,Pvt Ltd,1
freq,3510,10601,10942,11042,9242,11538,2673,2453,7838,6414


In [40]:
train.nunique()

enrollee_id               15326
city                        123
city_development_index       93
gender                        3
relevent_experience           2
enrolled_university           3
education_level               5
major_discipline              6
experience                   22
company_size                  8
company_type                  6
last_new_job                  6
training_hours              241
target                        2
dtype: int64

In [41]:
test.nunique()

enrollee_id               3832
city                       113
city_development_index      87
gender                       3
relevent_experience          2
enrolled_university          3
education_level              5
major_discipline             6
experience                  22
company_size                 8
company_type                 6
last_new_job                 6
training_hours             235
dtype: int64

In [42]:
train.isnull().sum()

enrollee_id                  0
city                         0
city_development_index       0
gender                    3576
relevent_experience          0
enrolled_university        314
education_level            365
major_discipline          2281
experience                  54
company_size              4787
company_type              4943
last_new_job               342
training_hours               0
target                       0
dtype: int64

In [43]:
test.isnull().sum()

enrollee_id                  0
city                         0
city_development_index       0
gender                     932
relevent_experience          0
enrolled_university         72
education_level             95
major_discipline           532
experience                  11
company_size              1151
company_type              1197
last_new_job                81
training_hours               0
dtype: int64

In [44]:
train['target'].value_counts()

target
0.0    11517
1.0     3809
Name: count, dtype: int64

### 기본 예측

### 데이터 전처리

#### 인코딩

In [45]:
target = train.pop('target')

#### 결측치 제거

In [46]:
train = train.fillna('X')
test = train.fillna('X')

In [47]:
data = pd.concat([train,test])
data = pd.get_dummies(data)

train = data[:len(train)].copy()
test = data[len(train):].copy()

print('train :', train.shape)
print('test :', test.shape)

train : (15326, 195)
test : (15326, 195)


#### 검증 데이터 나누기

In [48]:
x_train,x_val,y_train,y_val = train_test_split(
    train,
    target,
    test_size=0.2,
    random_state=0
)

print('x_train :', x_train.shape)
print('x_val :', x_val.shape)
print('y_train :', y_train.shape)
print('y_val :', y_val.shape)

x_train : (12260, 195)
x_val : (3066, 195)
y_train : (12260,)
y_val : (3066,)


#### 모델 학습

In [50]:
model = RandomForestClassifier(random_state=0)
model.fit(x_train,y_train)

RandomForestClassifier(random_state=0)

In [52]:
pred = model.predict_proba(x_val)
pred

array([[0.73, 0.27],
       [0.35, 0.65],
       [0.67, 0.33],
       ...,
       [0.6 , 0.4 ],
       [0.97, 0.03],
       [0.35, 0.65]])

#### 모델 평가

In [53]:
roc_auc = roc_auc_score(y_val, pred[:,1])
print("roc_auc :", roc_auc)

roc_auc : 0.7730742036233207


### 성능개선

In [136]:
train = pd.read_csv('hr_train.csv')
test = pd.read_csv('hr_test.csv')

#### 결측치 대치

In [137]:
train.describe(include='O')

,city,gender,relevent_experience,enrolled_university,education_level,major_discipline,experience,company_size,company_type,last_new_job
count,15326,11750,15326,15012,14961,13045,15272,10539,10383,14984
unique,123,3,2,3,5,6,22,8,6,6
top,city_103,Male,Has relevent experience,no_enrollment,Graduate,STEM,>20,50-99,Pvt Ltd,1
freq,3510,10601,10942,11042,9242,11538,2673,2453,7838,6414


In [138]:
null_cols = train.isnull().sum()
null_cols = null_cols[null_cols >0]
null_cols = null_cols.keys()
null_cols

Index(['gender', 'enrolled_university', 'education_level', 'major_discipline',
       'experience', 'company_size', 'company_type', 'last_new_job'],
      dtype='object')

In [139]:
mode_dic = {}

for col in null_cols:
    m = train[col].mode()[0]
    mode_dic[col] = m

mode_dic

{'gender': 'Male',
 'enrolled_university': 'no_enrollment',
 'education_level': 'Graduate',
 'major_discipline': 'STEM',
 'experience': '>20',
 'company_size': '50-99',
 'company_type': 'Pvt Ltd',
 'last_new_job': '1'}

In [140]:
train = train.fillna('X')
test = test.fillna('X')

#for key,value in mode_dic.items():
#    train[key] = train[key].fillna(value)
#    test[key] = test[key].fillna(value)

### 라벨인코딩

In [141]:
obj_cols = train.select_dtypes(include='O').columns
obj_cols

Index(['city', 'gender', 'relevent_experience', 'enrolled_university',
       'education_level', 'major_discipline', 'experience', 'company_size',
       'company_type', 'last_new_job'],
      dtype='object')

In [142]:
for col in obj_cols:
    le = LabelEncoder()
    train[col] = le.fit_transform(train[col])
    test[col] = le.transform(test[col])

#### 스케일링

In [143]:
scaler_dic = {
    "MinMaxScaler" : MinMaxScaler(),
    "StandardScaler" : StandardScaler(),
    "RobustScaler" : RobustScaler(),
    "MaxAbsScaler" : MaxAbsScaler(),
}

In [144]:
target = train.pop('target')

In [145]:
print(train.shape)
print(test.shape)

(15326, 13)
(3832, 13)


In [146]:
#결측치 대치가 완료된 데이터
origin_train = train.copy()
origin_test = test.copy()

for name, scaler in scaler_dic.items():
    
    train = origin_train.copy()   
    test = origin_test.copy()

    #스케일링 실행
    train = scaler.fit_transform(train)
    test = scaler.transform(test)

    #데이터 나누기
    x_train,x_val,y_train,y_val = train_test_split(
        train,
        target,
        test_size=0.2,
        random_state=0,
    )

    #모델생성
    model = RandomForestClassifier(random_state=0)
    model.fit(x_train,y_train)

    #예측
    pred = model.predict_proba(x_val)

    #평가
    roc_auc = roc_auc_score(y_val,pred[:,1])

    print(name,":",roc_auc)

MinMaxScaler : 0.7749741291715249
StandardScaler : 0.7749234996090111
RobustScaler : 0.7750549023464022
MaxAbsScaler : 0.7750660232907695


#### 하이퍼파라미터 튜닝

In [147]:
train = origin_train.copy()
test = origin_test.copy()

#스케일링 실행
scaler = RobustScaler()
train = scaler.fit_transform(train)
test = scaler.transform(test)

#데이터 나누기
x_train,x_val,y_train,y_val = train_test_split(
    train,
    target,
    test_size=0.2,
    random_state=0,
)

for depth in range(3,8):
    #모델생성
    model = RandomForestClassifier(max_depth=depth, random_state=0)
    model.fit(x_train,y_train)

    #예측
    pred = model.predict_proba(x_val)

    #평가
    roc_auc = roc_auc_score(y_val,pred[:,1])
    #depth({depth}) : {roc_auc}
    print(f'depth({depth}) : {roc_auc}')

depth(3) : 0.775941358675039
depth(4) : 0.777482487439186
depth(5) : 0.7796899948960718
depth(6) : 0.7816847411278277
depth(7) : 0.7816010413886431


In [150]:
#n_estimators 튜닝
for n in range(200, 600, 100):
    #모델생성
    model = RandomForestClassifier(max_depth=7, n_estimators=n, random_state=0)
    model.fit(x_train,y_train)

    #예측
    pred = model.predict_proba(x_val)

    #평가
    roc_auc = roc_auc_score(y_val,pred[:,1])
    print(f'n_estimators({n}) : {roc_auc}')

n_estimators(200) : 0.7825363713412095
n_estimators(300) : 0.7827131358253613
n_estimators(400) : 0.7825246650839807
n_estimators(500) : 0.7823490712255514
